In [ ]:
import os
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    CLIPModel,
    CLIPProcessor,
    get_linear_schedule_with_warmup
)

from tqdm.auto import tqdm

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/baohuynhlongthien/deepfashion-images"

CAPTION_PATH = "/kaggle/input/datasets/baohuynhlongthien/deepfashion-caption/captions_fixed.csv"

OUTPUT_DIR = "/kaggle/working/checkpoints"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv(CAPTION_PATH)

print(df.head())

print(df.shape)

In [ ]:
df["image_path"] = (
    df["image_path"]
    .str.replace("img/", "", regex=False)
    .str.replace(
        "Striped_A-line_Dress",
        "Striped_A-Line_Dress",
        regex=False
    )
)

In [ ]:
train_df = df[df["split"]=="train"].reset_index(drop=True)

val_df = df[df["split"]=="val"].reset_index(drop=True)

test_df = df[df["split"]=="test"].reset_index(drop=True)

print(len(train_df))
print(len(val_df))
print(len(test_df))

# **Load OpenAI CLIP**

In [ ]:
processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)

model.to(device)

# **Dataset**

In [ ]:
class DeepFashionDataset(Dataset):
    def __init__(self, dataframe, image_root, processor):
        self.df = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_rel_path = row["image_path"]

        if image_rel_path.startswith("img/"):
            image_rel_path = image_rel_path[4:]

        image_path = os.path.join(self.image_root, image_rel_path)
        
        caption = row["caption"]

        image = Image.open(image_path).convert("RGB")

        encoding = self.processor(
            text=caption,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True
        )

        item = {
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

        return item

In [ ]:
train_dataset = DeepFashionDataset(
    train_df,
    DATA_ROOT,
    processor
)

val_dataset = DeepFashionDataset(
    val_df,
    DATA_ROOT,
    processor
)

test_dataset = DeepFashionDataset(
    test_df,
    DATA_ROOT,
    processor
)

# **DataLoader**

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 1e-6
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-6,
    weight_decay=0.01
)

In [ ]:
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=500,
    num_training_steps=len(train_loader)*EPOCHS
)

In [ ]:
best_val_loss = float("inf")

train_losses = []
val_losses = []

# **Training**

In [ ]:
from tqdm import tqdm

for epoch in range(EPOCHS):

    # Train
    model.train()
    running_train_loss = 0.0

    for batch in tqdm(train_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_loss=True
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        scheduler.step()

        running_train_loss += loss.item()

    train_loss = running_train_loss / len(train_loader)
    train_losses.append(train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                return_loss=True
            )

            running_val_loss += outputs.loss.item()

    val_loss = running_val_loss / len(val_loader)
    val_losses.append(val_loss)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_clip_model.pt")
        print("Best model saved.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curve")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import pandas as pd

META_PATH = "/kaggle/input/datasets/baohuynhlongthien/format/metadata_clean.csv"

meta = pd.read_csv(META_PATH)

print("Metadata người 1:")
print(meta.shape)
print(meta.columns.tolist())

df_fixed = df.copy()

df_fixed["image_path"] = "img/" + df_fixed["image_path"]

print("\nMy dataframe:")
print(df_fixed.shape)
print(df_fixed.columns.tolist())

display(df_fixed.head())

In [ ]:
my_folders = set(
    df_fixed["image_path"]
    .str.extract(r"img/(.*?)/")[0]
    .dropna()
    .unique()
)

his_folders = set(
    meta["image_path"]
    .str.extract(r"img/(.*?)/")[0]
    .dropna()
    .unique()
)

only_mine = my_folders - his_folders
only_his = his_folders - my_folders

print("Only mine :", len(only_mine))
print("Only his  :", len(only_his))

In [ ]:
from difflib import SequenceMatcher

his_list = sorted(only_his)

mapping = {}

for my_name in sorted(only_mine):

    best = None
    best_score = 0

    for his_name in his_list:

        score = SequenceMatcher(
            None,
            my_name,
            his_name
        ).ratio()

        if score > best_score:
            best_score = score
            best = his_name

    mapping[my_name] = (best, best_score)

print("Total mappings:", len(mapping))

In [ ]:
for mine, (his, score) in list(mapping.items())[:50]:

    print(f"{score:.3f}   {mine}")
    print(f"        -> {his}")
    print()

In [ ]:
for mine, (his, score) in mapping.items():

    if score >= 0.90:

        df_fixed["image_path"] = (
            df_fixed["image_path"]
            .str.replace(
                mine,
                his,
                regex=False
            )
        )

print("Mapping applied.")

display(df_fixed.head())

In [ ]:
df_export = meta.merge(
    df_fixed[["image_path", "caption"]],
    on="image_path",
    how="left"
)

print(len(df_export))
print(df_export["caption"].isna().sum())

display(df_export.head())

In [ ]:
print(
    meta["image_path"].duplicated().sum()
)

In [ ]:
print(
    df_fixed["image_path"].duplicated().sum()
)

In [ ]:
df_fixed = df_fixed.drop_duplicates(
    subset=["image_path"],
    keep="first"
)

In [ ]:
df_export = meta.merge(
    df_fixed[["image_path", "caption"]],
    on="image_path",
    how="left"
)

print(len(df_export))

In [ ]:
import torch

checkpoint = torch.load(
    "/kaggle/input/datasets/baohuynhlongthien/checkpoint/best_clip_model.pt",
    map_location=device
)

model.load_state_dict(checkpoint)

model.eval()

print("Best model loaded.")

In [ ]:
reverse_mapping = {}

for mine, (his, score) in mapping.items():
    reverse_mapping[his] = mine

print(len(reverse_mapping))

In [ ]:
df_export["real_image_path"] = df_export["image_path"]

for his, mine in reverse_mapping.items():

    df_export["real_image_path"] = (
        df_export["real_image_path"]
        .str.replace(
            his,
            mine,
            regex=False
        )
    )

display(
    df_export[
        ["image_path", "real_image_path"]
    ].head(20)
)

In [ ]:
batch = df_export.iloc[:2]

images = []

for _, row in batch.iterrows():

    image_path = os.path.join(
        DATA_ROOT,
        row["real_image_path"].replace("img/", "")
    )

    images.append(Image.open(image_path).convert("RGB"))

inputs = processor(images=images, return_tensors="pt").to(device)

vision_outputs = model.get_image_features(**inputs)

print(type(vision_outputs))
print(type(vision_outputs.pooler_output))
print(vision_outputs.pooler_output.shape)

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

model.eval()

records = df_export.to_dict("records")

image_embeddings = []

with torch.no_grad():

    for row in tqdm(records):

        image_path = os.path.join(
            DATA_ROOT,
            row["real_image_path"].replace("img/", "")
        )

        image = Image.open(image_path).convert("RGB")

        inputs = processor(
            images=image,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        outputs = model.get_image_features(**inputs)

        if hasattr(outputs, "pooler_output"):
            features = outputs.pooler_output
        else:
            features = outputs

        features = F.normalize(
            features,
            p=2,
            dim=-1
        )

        image_embeddings.append(
            features.squeeze(0)
            .cpu()
            .numpy()
            .astype(np.float16)
        )

image_embeddings = np.stack(image_embeddings)

print(image_embeddings.shape)
print(image_embeddings.dtype)
print(np.linalg.norm(image_embeddings[0]))

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

model.eval()

text_embeddings = []

with torch.no_grad():

    for caption in tqdm(df_export["caption"]):

        inputs = processor(
            text=caption,
            return_tensors="pt",
            truncation=True,
            padding=True
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        outputs = model.get_text_features(**inputs)

        if hasattr(outputs, "pooler_output"):
            features = outputs.pooler_output
        else:
            features = outputs

        features = F.normalize(
            features,
            p=2,
            dim=-1
        )

        text_embeddings.append(
            features.squeeze(0)
            .cpu()
            .numpy()
            .astype(np.float16)
        )

text_embeddings = np.stack(text_embeddings)

print(text_embeddings.shape)
print(text_embeddings.dtype)
print(np.linalg.norm(text_embeddings[0]))

In [ ]:
import numpy as np

np.save(
    "image_embeddings.npy",
    image_embeddings
)

np.save(
    "text_embeddings.npy",
    text_embeddings
)

metadata_clean = df_export[
    ["image_path", "category", "split"]
]

metadata_clean.to_csv(
    "metadata_clean.csv",
    index=False
)

print("Done!")

In [ ]:
print("Image:", image_embeddings.shape, image_embeddings.dtype)
print("Text :", text_embeddings.shape, text_embeddings.dtype)
print("Metadata:", metadata_clean.shape)

print("Image norm:", np.linalg.norm(image_embeddings[0]))
print("Text norm :", np.linalg.norm(text_embeddings[0]))